In [ ]:
# install numpy torch transformers datasets trl
import re
import json
from datetime import datetime
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

# Fine tuning:

First create the model:

In [ ]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

Then create the dataset:

In [ ]:
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en"
)

dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
def preprocess(example):
    return {
        "messages": [
            {"role": "user", "content": example["Question"]},
            {
                "role": "assistant",
                "content": f"<think>{example['Complex_CoT']}</think>\n\n{example['Response']}"
                # "content": f"{example['Response']}"
            }
        ]
    }

dataset = dataset.map(
    preprocess,
    remove_columns=["Question", "Response", "Complex_CoT"]
)

In [ ]:
# We split the training dataset 80/20
TRAIN_DATASET_LENGTH = len(dataset["train"])
TRAIN_DATASET_SPLITTER = int(0.8*TRAIN_DATASET_LENGTH)
dataset_train80 = dataset["train"].shuffle(seed=42).select(range(TRAIN_DATASET_LENGTH)[:TRAIN_DATASET_SPLITTER])
dataset_train20 = dataset["train"].shuffle(seed=42).select(range(TRAIN_DATASET_LENGTH)[TRAIN_DATASET_SPLITTER:])

Set the training parameters:

In [ ]:
def compute_metrics(eval_pred):
    losses = eval_pred.losses

    mean_loss = np.mean(losses)
    perplexity = np.exp(mean_loss)

    return {"perplexity": perplexity}

def preprocess_logits_for_metrics(logits, labels):
    # somehow prevents memory leak (i think), be careful with it
    # predictions = torch.argmax(logits, dim=-1)
    predictions = None
    return predictions

In [ ]:
config80 = SFTConfig(
    output_dir="./smollm2-135m-sft-adaptative80-training-checkpoints",
    bf16=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    # torch_empty_cache_steps=20,
    learning_rate=1e-4,
    num_train_epochs=2.0,
    logging_steps=0.02,
    eval_strategy="steps",
    # eval_steps=0.2,
    save_steps=0.2,
    save_total_limit=2,
    report_to="none",
    include_for_metrics=["loss"],
    seed=42,
)

In [ ]:
trainer80 = SFTTrainer(
    model=model,
    args=config80,
    processing_class=tokenizer,
    train_dataset=dataset_train80,
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

In [ ]:
trainer80.train()

In [ ]:
trainer80.evaluate()

In [ ]:
config20 = SFTConfig(
    output_dir="./smollm2-135m-sft-adaptative20-training-checkpoints",
    bf16=False,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    # torch_empty_cache_steps=20,
    learning_rate=1e-4,
    num_train_epochs=2.0,
    logging_steps=0.02,
    eval_strategy="steps",
    # eval_steps=0.2,
    save_steps=0.2,
    save_total_limit=2,
    report_to="none",
    include_for_metrics=["loss"],
    seed=42,
)

In [ ]:
trainer20 = SFTTrainer(
    model=model,
    args=config20,
    processing_class=tokenizer,
    train_dataset=dataset_train20,
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

In [ ]:
trainer20.train()

In [ ]:
trainer20.evaluate()

Save the fine tuned model:

In [ ]:
model_name = "SmolLM2-135M-SFT-adaptative"
model.save_pretrained(model_name)

# Visual test:

Select an example from the test dataset:

In [ ]:
ds_example = dataset["test"][0]["messages"][0]["content"]

messages = [
    {"role": "user", "content": ds_example},
]

Generate output for the model from the example:

In [ ]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

outputs = model.generate(**inputs, max_new_tokens=1000)

Then see answer:

In [ ]:
# Example question
print(ds_example)

In [ ]:
# Fine tuned model answer
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

# Benchmark:

If you already have a fine tuned model, execute this cell:

In [ ]:
# If the model is already trained, load it (change the model name)
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
model_name = "SmolLM2-135M-SFT-adaptative"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(f"./{model_name}/", local_files_only=True).to(device)

Create the dataset for the benchmark:

In [ ]:
# Load database
medqa_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")

In [ ]:
# Process database
def medqa_prompt(example):
    options = example["options"]
    prompt = (
        "You are a medical expert. Choose the single best answer.\n\n"
        f"Question:\n{example['question']}\n\n"
        "Options:\n"
        f"A. {options['A']}\n"
        f"B. {options['B']}\n"
        f"C. {options['C']}\n"
        f"D. {options['D']}\n\n"
        "Answer with the letter (A, B, C, or D)."
    )
    return prompt

def medqa_preprocess(example):
    return {
        "messages": [
            {"role": "user", "content": medqa_prompt(example)}
        ]
    }

medqa_dataset = medqa_dataset.map(
    medqa_preprocess,
    remove_columns=["question", "answer", "options", "meta_info", "metamap_phrases"]
)

In [ ]:
# small_medqa = medqa_dataset.shuffle(seed=42).select(range(50))

## Single question visual test (facultative):

Facultative test to see if everything works. Select an example:

In [ ]:
medqa_example_question = medqa_dataset[0]["messages"][0]["content"]
medqa_example_answer = medqa_dataset[0]["answer_idx"]

medqa_messages = [
    {"role": "user", "content": medqa_example_question},
]

Generate the output:

In [ ]:
medqa_inputs = tokenizer.apply_chat_template(
	medqa_messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(device)

medqa_outputs = model.generate(
    **medqa_inputs,
    max_new_tokens=1000,
    do_sample=False,        # GREEDY
    temperature=0.0,        # DETERMINISTIC
    top_p=1.0,
)

See answer:

In [ ]:
# Example question
print(medqa_example_question)

In [ ]:
# Example ground truth answer
print(medqa_example_answer)

In [ ]:
# Fine tuned model answer
print(tokenizer.decode(medqa_outputs[0][medqa_inputs["input_ids"].shape[-1]:]))

## Actual benchmarking:

Some functions to evaluate models on the benchmark dataset:

In [ ]:
def get_output(messages, model, tokenizer):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False,        # GREEDY
        temperature=0.0,        # DETERMINISTIC
        top_p=1.0,
    )
    output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    return output

In [ ]:
def extract_choice(text):
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    matches = re.findall(r"\b([ABCD])\b", text)
    return matches[-1] if matches else None

In [ ]:
def evaluate_on(dataset, model, tokenizer, from_row=0, save_steps=100, save_file='default_save', verbose=0):
  correct = 0
  total = 0
  invalid = 0
  i = from_row

  for row in dataset.select(range(from_row, len(dataset))):
    if verbose>0:
      print(f"\nRow {i}: {str(datetime.now())}")

    output = get_output(row["messages"], model, tokenizer)
    if verbose>2:
      print(f"Post output: {str(datetime.now())}")

    choice = extract_choice(output)
    if verbose>2:
      print(f"Post choice: {str(datetime.now())}")

    answer = row["answer_idx"]

    if choice is None:
      invalid += 1
      if verbose>1:
        print(f"Invalid inference: {output}")
    else:
      correct += int(choice == answer)
    total += 1

    if (i+1) % save_steps == 0 or (i+1) == len(dataset): # step = i+1
      accuracy = correct / total
      invalid_rate = invalid / total
      dict_to_save = {
          "accuracy": accuracy,
          "invalid_rate": invalid_rate,
          "correct": correct,
          "total": total,
          "invalid": invalid,
          "from_row": from_row,
          "last_row": i,
      }
      with open(f"{save_file}_row{i}.json", 'w') as jf:
        json.dump(dict_to_save, jf)
      print(f"Checkpoint saved at '/{save_file}_row{i}.json'")

    i += 1 # increment at the end of the loop, meaning i is the row number starting from 0 to len(dataset)-1

    if verbose>2:
      print(f"Post total: {str(datetime.now())}")

  accuracy = correct / total
  invalid_rate = invalid / total

  return {
      "accuracy": accuracy,
      "invalid_rate": invalid_rate,
      "correct": correct,
      "total": total,
      "invalid": invalid,
      "from_row": from_row,
  }

Run the stuff. If GPU runs out, set ```from_row=<last_row>+1```, with ```last_row``` from your latest save, you will have to sum the result manually though. Good luck:

In [ ]:
save_file = "medqa_results_sft_adaptative"
medqa_results_sft_adaptative = evaluate_on(medqa_dataset, model, tokenizer, save_steps=250, save_file=save_file, verbose=2)

In [ ]:
print(medqa_results_sft_adaptative)